In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis, skew
from scipy.signal import welch, find_peaks
from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import tensorflow as tf

from statsmodels.tsa.arima.model import ARIMA

tf.random.set_seed(42)
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3
})

TRAFFIC_LABELS = {
    0: "Empty Bridge", 1: "Light Traffic",
    2: "Moderate Traffic", 3: "Heavy Traffic",
    4: "Single Heavy Veh.", 5: "Multi Heavy Veh.",
    6: "Peak Hour Traffic", 7: "Night Traffic"
}

N_SENSORS = 6
SENSOR_NAMES = [f"Sensor {i+1}" for i in range(N_SENSORS)]
COLORS = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0','#00BCD4','#FF5722','#607D8B']
SENSOR_COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

print("="*68)
print("  BRIDGE TRAFFIC CONDITION CLASSIFICATION")
print("  6-Sensor Vibration Analysis | RF + ARIMA + LSTM")
print("="*68)

print("\n[1/8] Loading dataset...")

folder = "/content"
files = sorted([f for f in os.listdir(folder) if f.endswith(".txt")])

all_data = []
labels_raw = []

for i, file in enumerate(files):
    rows = []
    with open(os.path.join(folder, file)) as f:
        for idx, line in enumerate(f):
            if idx < 5:
                continue
            line = line.strip()
            if not line:
                continue
            try:
                vals = [float(v) for v in line.split('\t') if v.strip()]
                if len(vals) >= N_SENSORS:
                    rows.append(vals[:N_SENSORS])
                elif len(vals) > 0:
                    rows.append(vals)
            except ValueError:
                continue
    if rows:
        df = pd.DataFrame(rows, columns=SENSOR_NAMES[:len(rows[0])])
        all_data.append(df)
        labels_raw.append(i)

n_classes = len(all_data)
actual_sensors = all_data[0].shape[1]
SENSOR_NAMES = [f"Sensor {i+1}" for i in range(actual_sensors)]

print(f"\n  {'File':<14} {'Condition':<25} {'Rows':>8}  {'Sensors':>8}")
print("  " + "-"*58)

for i, (f, df) in enumerate(zip(files, all_data)):
    print(f"  {f:<14} {TRAFFIC_LABELS[i]:<25} {len(df):>8,}  {df.shape[1]:>8}")

print(f"\n  Total conditions : {n_classes}")
print(f"  Sensors per file : {actual_sensors}")
print(f"  Total rows       : {sum(len(d) for d in all_data):,}")

print(f"\n  Sample data (first 5 rows, Condition 0 — {TRAFFIC_LABELS[0]}):")
print(all_data[0].head().to_string())

print("\n[2/8] Exploratory Data Analysis...")

data_norm = []
for df in all_data:
    normed = (df - df.mean()) / (df.std() + 1e-8)
    data_norm.append(normed)

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
fig.suptitle("Bridge Vibration Signals — All 6 Sensors per Traffic Condition", fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    seg = data_norm[i].iloc[:800]
    for j, col in enumerate(seg.columns):
        ax.plot(seg[col].values, color=SENSOR_COLORS[j], lw=0.65, alpha=0.85)
    ax.set_title(f"Class {i}: {TRAFFIC_LABELS[i]}", fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig("eda_01_all_sensors.png", dpi=120, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Per-Sensor Signal — Comparison Across All Traffic Conditions", fontsize=12, fontweight='bold')

for s_idx, ax in enumerate(axes.flat):
    if s_idx >= actual_sensors:
        break
    for c_idx in range(n_classes):
        seg = data_norm[c_idx].iloc[:600, s_idx].values
        ax.plot(seg, color=COLORS[c_idx], lw=0.7, alpha=0.75)

plt.tight_layout()
plt.savefig("eda_02_sensor_comparison.png", dpi=120, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Sensor Correlation Heatmap per Traffic Condition", fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.flat):
    corr = data_norm[i].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)

plt.tight_layout()
plt.savefig("eda_03_sensor_correlations.png", dpi=120, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(4, 2, figsize=(14, 10))
fig.suptitle("FFT Frequency Spectrum — Sensor 1, All Traffic Conditions", fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.flat):
    seg = data_norm[i].iloc[:4096, 0].values
    N = len(seg)
    mag = np.abs(rfft(seg))[:N//4]
    frq = rfftfreq(N)[:N//4]
    ax.plot(frq, mag)

plt.tight_layout()
plt.savefig("eda_04_fft_spectra.png", dpi=120, bbox_inches='tight')
plt.close()

stats_rows = []

for i, df in enumerate(data_norm):
    for col in df.columns:
        seg = df[col].values[:8000]
        freqs, psd = welch(seg, nperseg=256)
        stats_rows.append({
            'Condition': TRAFFIC_LABELS[i],
            'Sensor': col,
            'Mean': np.mean(seg),
            'Std': np.std(seg),
            'RMS': np.sqrt(np.mean(seg**2)),
            'Peak': np.max(np.abs(seg)),
            'Kurt': kurtosis(seg),
            'Skew': skew(seg)
        })

stats_df = pd.DataFrame(stats_rows)

agg_stats = stats_df.groupby('Condition')[['RMS','Peak','Kurt','Skew']].mean()

rms_matrix = np.zeros((n_classes, actual_sensors))

for i, df in enumerate(data_norm):
    for j, col in enumerate(df.columns):
        rms_matrix[i, j] = np.sqrt(np.mean(df[col].values**2))

WINDOW = 200
STRIDE = 100

def extract_window_features(window_2d):
    feats = []
    n_sens = window_2d.shape[1]

    for s in range(n_sens):
        x = window_2d[:, s]
        feats += [
            np.mean(x),
            np.std(x),
            np.sqrt(np.mean(x**2)),
            np.max(np.abs(x)),
            kurtosis(x),
            skew(x)
        ]

    return np.array(feats, dtype=np.float32)

X_feat_list, X_seq_list, y_list = [], [], []

for cond_idx, df in enumerate(data_norm):
    arr = df.values.astype(np.float32)
    for start in range(0, len(arr) - WINDOW, STRIDE):
        win = arr[start:start+WINDOW]
        X_feat_list.append(extract_window_features(win))
        X_seq_list.append(win)
        y_list.append(cond_idx)

X_feat = np.array(X_feat_list)
X_seq = np.array(X_seq_list)
y_all = np.array(y_list)

X_tr_feat, X_te_feat, X_tr_seq, X_te_seq, y_tr, y_te = train_test_split(
    X_feat, X_seq, y_all, test_size=0.2, random_state=42, stratify=y_all
)

scaler = StandardScaler()
X_tr_feat = scaler.fit_transform(X_tr_feat)
X_te_feat = scaler.transform(X_te_feat)

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_tr_feat, y_tr)

rf_pred = rf.predict(X_te_feat)

print(classification_report(y_te, rf_pred))

  BRIDGE TRAFFIC CONDITION CLASSIFICATION
  6-Sensor Vibration Analysis | RF + ARIMA + LSTM

[1/8] Loading dataset...

  File           Condition                     Rows   Sensors
  ----------------------------------------------------------
  test1.txt      Empty Bridge                11,137         6
  test2.txt      Light Traffic                7,105         6
  test3.txt      Moderate Traffic            13,921         6
  test4.txt      Heavy Traffic                3,361         6
  test5.txt      Single Heavy Veh.            7,809         6
  test6.txt      Multi Heavy Veh.             4,129         6
  test7.txt      Peak Hour Traffic           12,289         6
  test8.txt      Night Traffic                6,593         6

  Total conditions : 8
  Sensors per file : 6
  Total rows       : 66,344

  Sample data (first 5 rows, Condition 0 — Empty Bridge):
   Sensor 1  Sensor 2  Sensor 3  Sensor 4  Sensor 5  Sensor 6
0     0.000  0.010769  0.018895  0.017099  0.012776  0.002928
1   